In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
import yake
import spacy
import pytextrank
import torch
from tqdm import tqdm

C:\Users\kinge\anaconda3\Lib\site-packages


In [2]:
df = pd.read_csv('../data/04_input_keyword_extraction.csv')

In [3]:
kw_extractor = yake.KeywordExtractor(lan="en", n=2, top=10)
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("textrank")

def extract_yake(text):
    if pd.isna(text) or not text: return []
    keywords = kw_extractor.extract_keywords(str(text))
    return [kw[0] for kw in keywords]

def extract_textrank(text):
    if pd.isna(text) or not text: return []
    doc = nlp(str(text))
    return [phrase.text for phrase in doc._.phrases[:5]]

df['yake_keywords'] = df['grievance_text_processed'].apply(extract_yake)
df['textrank_keywords'] = df['grievance_text_processed'].apply(extract_textrank)

In [4]:
import nltk
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

def extractive_summarize(text, sentence_count=2):
    if not text or len(text.split()) < 20: return text
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = LexRankSummarizer()
    summary = summarizer(parser.document, sentence_count)
    return " ".join([str(sentence) for sentence in summary])

df['extractive_summary'] = df['grievance_text_processed'].apply(extractive_summarize)

In [5]:
df.to_csv('../data/05_input_risk_scoring.csv', index=False)